In [1]:
import numpy as np

In [2]:
from sklearn.svm import SVC,SVR
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neighbors import KNeighborsClassifier,KNeighborsRegressor
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.metrics import accuracy_score, r2_score
from sklearn.ensemble import VotingClassifier,VotingRegressor
from sklearn.datasets import load_diabetes, load_digits

# Voting Ensemble
- base model algoridums are diffrent
- Same dataset to train all those mode

## Assumption
- base model algo should be indipandent
- minimum accuracy < 51%

## Classification

In [3]:
X_clf,y_clf = load_digits(return_X_y=True)
X_train_clf,X_test_clf,y_train_clf,y_test_clf = train_test_split(X_clf,y_clf,train_size=.7)

In [4]:
estimaters = [
    ('svc',SVC()),
    ('lr_clf',LogisticRegression(max_iter=200)),
    ('dtree_clf',DecisionTreeClassifier()),
    ('knn_clf',KNeighborsClassifier())
]
for m in estimaters:
    model = m[1]
    model.fit(X_train_clf,y_train_clf)
    y_pred = model.predict(X_test_clf)
    print(m[0], np.round(accuracy_score(y_pred=y_pred,y_true=y_test_clf),2))
    print('Cross Val Score',np.mean(cross_val_score(model,X_clf,y_clf,cv=10,scoring='accuracy')).round(2))

svc 0.98
Cross Val Score 0.97
lr_clf 0.97
Cross Val Score 0.93
dtree_clf 0.87
Cross Val Score 0.82
knn_clf 0.98
Cross Val Score 0.97


In [5]:
voting = VotingClassifier(estimators=estimaters,voting='hard')
voting.fit(X_train_clf,y_train_clf)

VotingClassifier(estimators=[('svc', SVC()),
                             ('lr_clf', LogisticRegression(max_iter=200)),
                             ('dtree_clf', DecisionTreeClassifier()),
                             ('knn_clf', KNeighborsClassifier())])

In [6]:
y_pred = voting.predict(X_test_clf)
print('Voting Ensemble Classification', np.round(accuracy_score(y_pred=y_pred,y_true=y_test_clf),2))
print('Cross Val Score',np.mean(cross_val_score(voting,X_clf,y_clf,cv=10,scoring='accuracy')).round(2))

Voting Ensemble Classification 0.98
Cross Val Score 0.96


In [7]:
combo_clf:dict = {}
for w1 in range(5):
    for w2 in range(5):
        for w3 in range(5):
            for w4 in range(5):
                if [w1,w2,w3,w4] == [0,0,0,0]:
                    continue
                voting = VotingClassifier(estimators=estimaters,voting='hard',weights=[w1,w2,w3,w4])
                voting.fit(X_train_clf,y_train_clf)
                y_pred = voting.predict(X_test_clf)
                print(f'weights [{w1},{w2},{w3},{w4}]')
                print('Voting Ensemble Classification', np.round(accuracy_score(y_pred=y_pred,y_true=y_test_clf),2))
                cvs = np.mean(cross_val_score(voting,X_clf,y_clf,cv=5,scoring='accuracy'))
                print('Cross Val Score',cvs,end='\n\n')
                combo_clf[cvs] = [w1,w2,w3,w4]

weights [0,0,0,1]
Voting Ensemble Classification 0.98
Cross Val Score 0.9627282575054161

weights [0,0,0,2]
Voting Ensemble Classification 0.98
Cross Val Score 0.9627282575054161

weights [0,0,0,3]
Voting Ensemble Classification 0.98
Cross Val Score 0.9627282575054161

weights [0,0,0,4]
Voting Ensemble Classification 0.98
Cross Val Score 0.9627282575054161

weights [0,0,1,0]
Voting Ensemble Classification 0.87
Cross Val Score 0.7757768492726711

weights [0,0,1,1]
Voting Ensemble Classification 0.9
Cross Val Score 0.8648142989786443

weights [0,0,1,2]
Voting Ensemble Classification 0.98
Cross Val Score 0.9627282575054161

weights [0,0,1,3]
Voting Ensemble Classification 0.98
Cross Val Score 0.9627282575054161

weights [0,0,1,4]
Voting Ensemble Classification 0.98
Cross Val Score 0.9627282575054161

weights [0,0,2,0]
Voting Ensemble Classification 0.84
Cross Val Score 0.7724481584648716

weights [0,0,2,1]
Voting Ensemble Classification 0.86
Cross Val Score 0.7807892293407613

weights [0,

In [8]:
np.max(list(combo_clf.keys())), combo_clf[np.max(list(combo_clf.keys()))]

(np.float64(0.9666202414113277), [3, 0, 2, 3])

## Regression

In [9]:
X_rgs,y_rgs = load_diabetes(return_X_y=True)
X_train_rgs,X_test_rgs,y_train_rgs,y_test_rgs = train_test_split(X_rgs,y_rgs,train_size=.7)

In [10]:
estimaters = [
    ('svr',SVR()),
    ('lr_rgs',LinearRegression()),
    ('dtree_rgs',DecisionTreeRegressor(max_depth=4)),
    ('knn_rgs',KNeighborsRegressor(weights='distance'))
]
for m in estimaters:
    model = m[1]
    model.fit(X_train_rgs,y_train_rgs)
    y_pred = model.predict(X_test_rgs)
    print(m[0], np.round(r2_score(y_pred=y_pred,y_true=y_test_rgs),2))
    print('Cross Val Score',np.mean(cross_val_score(model,X_rgs,y_rgs,cv=10,scoring='r2')).round(2))

svr 0.17
Cross Val Score 0.15
lr_rgs 0.5
Cross Val Score 0.46
dtree_rgs 0.39
Cross Val Score 0.29
knn_rgs 0.45
Cross Val Score 0.34


In [11]:
voting = VotingRegressor(estimators=estimaters)
voting.fit(X_train_rgs,y_train_rgs)

VotingRegressor(estimators=[('svr', SVR()), ('lr_rgs', LinearRegression()),
                            ('dtree_rgs', DecisionTreeRegressor(max_depth=4)),
                            ('knn_rgs',
                             KNeighborsRegressor(weights='distance'))])

In [12]:
y_pred = voting.predict(X_test_rgs)
print('Voting Ensemble Regression', np.round(r2_score(y_pred=y_pred,y_true=y_test_rgs),2))
print('Cross Val Score',np.mean(cross_val_score(voting,X_rgs,y_rgs,cv=10,scoring='r2')).round(2))

Voting Ensemble Regression 0.5
Cross Val Score 0.42


In [13]:
combo_rgs:dict = {}
for w1 in range(5):
    for w2 in range(5):
        for w3 in range(5):
            for w4 in range(5):
                if [w1,w2,w3,w4] == [0,0,0,0]:
                    continue
                voting = VotingRegressor(estimators=estimaters,weights=[w1,w2,w3,w4])
                voting.fit(X_train_rgs,y_train_rgs)
                y_pred = voting.predict(X_test_rgs)
                print(f'weights [{w1},{w2},{w3},{w4}]')
                print('Voting Ensemble Regression', np.round(r2_score(y_pred=y_pred,y_true=y_test_rgs),2))
                cvs = np.mean(cross_val_score(voting,X_rgs,y_rgs,cv=5,scoring='r2'))
                print('Cross Val Score',cvs,end='\n\n')
                combo_rgs[cvs] = [w1,w2,w3,w4]

weights [0,0,0,1]
Voting Ensemble Regression 0.45
Cross Val Score 0.3820057119392716

weights [0,0,0,2]
Voting Ensemble Regression 0.45
Cross Val Score 0.3820057119392716

weights [0,0,0,3]
Voting Ensemble Regression 0.45
Cross Val Score 0.3820057119392716

weights [0,0,0,4]
Voting Ensemble Regression 0.45
Cross Val Score 0.3820057119392716

weights [0,0,1,0]
Voting Ensemble Regression 0.4
Cross Val Score 0.2823261079591132

weights [0,0,1,1]
Voting Ensemble Regression 0.49
Cross Val Score 0.4072213186310666

weights [0,0,1,2]
Voting Ensemble Regression 0.49
Cross Val Score 0.4130911482749605

weights [0,0,1,3]
Voting Ensemble Regression 0.48
Cross Val Score 0.41199229018560046

weights [0,0,1,4]
Voting Ensemble Regression 0.48
Cross Val Score 0.4073515743109632

weights [0,0,2,0]
Voting Ensemble Regression 0.4
Cross Val Score 0.28806792933815933

weights [0,0,2,1]
Voting Ensemble Regression 0.47
Cross Val Score 0.3777061904384776

weights [0,0,2,2]
Voting Ensemble Regression 0.49
Cros

In [14]:
np.max(list(combo_rgs.keys())), combo_rgs[np.max(list(combo_rgs.keys()))]

(np.float64(0.48822025341737146), [0, 4, 0, 1])